# Model Training with MLflow Experiment Tracking

This notebook trains all recommendation models and logs experiments to MLflow:
1. Collaborative Filtering (KNN — item-based & user-based)
2. Matrix Factorization (SVD)
3. Content-Based Filtering
4. Neural Collaborative Filtering (PyTorch)
5. Hybrid Ensemble

Each model's parameters, metrics, and artifacts are tracked in MLflow.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
from scipy.sparse import csr_matrix

from src.data_loader import RetailRocketDataLoader
from src.feature_engine import FeatureEngine
from src.models.collaborative import CollaborativeFilteringKNN
from src.models.matrix_factor import MatrixFactorizationModel
from src.models.content_based import ContentBasedModel
from src.models.ncf import NCFTrainer
from src.models.hybrid import HybridRecommender
from src.evaluation import RecommendationEvaluator
from src.utils import load_config, ensure_dir

import os
PROJECT_ROOT = os.path.abspath('..')
config = load_config(os.path.join(PROJECT_ROOT, 'configs/config.yaml'))
MODEL_DIR = ensure_dir(os.path.join(PROJECT_ROOT, 'models'))

mlflow.set_tracking_uri(os.path.join(PROJECT_ROOT, config['mlflow']['tracking_uri']))
mlflow.set_experiment(config['mlflow']['experiment_name'])

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {config['mlflow']['experiment_name']}")

## 1. Load & Prepare Data

In [ ]:
loader = RetailRocketDataLoader()
loader.run_pipeline()

train, val, test = loader.temporal_train_test_split()
interaction_matrix = loader.build_interaction_matrix()
item_features = loader.build_item_features()

n_users = len(loader.user_to_idx)
n_items = len(loader.item_to_idx)

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

evaluator = RecommendationEvaluator(top_k_values=config['evaluation']['top_k'])

## 2. Collaborative Filtering (Item-Based KNN)

In [ ]:
cf_config = config['models']['collaborative_filtering']

with mlflow.start_run(run_name='collaborative_filtering_item_knn'):
    cf_model = CollaborativeFilteringKNN(
        k=cf_config['k_neighbors'],
        metric=cf_config['similarity'],
        min_k=cf_config['min_k'],
        mode='item'
    )
    
    # Build train-only interaction matrix
    train_matrix = csr_matrix(
        (train['rating'].values, (train['user_idx'].values, train['item_idx'].values)),
        shape=(n_users, n_items)
    )
    cf_model.fit(train_matrix)
    
    # Log parameters
    mlflow.log_params(cf_model.get_params())
    
    # Evaluate
    cf_results = evaluator.evaluate_model(
        cf_model, test, n_items, model_name='CF-KNN',
        user_idx_col='user_idx', use_idx=True
    )
    
    # Log metrics
    for metric_name, value in cf_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(metric_name.replace('@', '_at_'), value)
    
    # Save model
    with open(os.path.join(MODEL_DIR, 'collaborative_filtering.pkl'), 'wb') as f:
        pickle.dump(cf_model, f)
    mlflow.log_artifact(os.path.join(MODEL_DIR, 'collaborative_filtering.pkl'))
    
    print("\nCF-KNN Results:")
    for k, v in sorted(cf_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")

## 3. Matrix Factorization (SVD)

In [ ]:
mf_config = config['models']['matrix_factorization']

with mlflow.start_run(run_name='matrix_factorization_svd'):
    mf_model = MatrixFactorizationModel(
        algorithm=mf_config['algorithm'],
        n_factors=mf_config['n_factors'],
        n_epochs=mf_config['n_epochs'],
        lr_all=mf_config['lr_all'],
        reg_all=mf_config['reg_all']
    )
    mf_model.fit(train)
    
    mlflow.log_params(mf_model.get_params())
    
    cv_results = mf_model.cross_validate(train, cv=3)
    for k, v in cv_results.items():
        mlflow.log_metric(f'cv_{k}', v)
    print(f"Cross-validation: {cv_results}")
    
    mf_results = evaluator.evaluate_model(
        mf_model, test, n_items, model_name='MF-SVD',
        use_idx=False
    )
    
    for metric_name, value in mf_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(metric_name.replace('@', '_at_'), value)
    
    with open(os.path.join(MODEL_DIR, 'matrix_factorization.pkl'), 'wb') as f:
        pickle.dump(mf_model, f)
    mlflow.log_artifact(os.path.join(MODEL_DIR, 'matrix_factorization.pkl'))
    
    print("\nMF-SVD Results:")
    for k, v in sorted(mf_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")

## 4. Content-Based Filtering

In [ ]:
with mlflow.start_run(run_name='content_based'):
    cb_model = ContentBasedModel(
        similarity_metric=config['models']['content_based']['similarity_metric']
    )
    cb_model.fit(train, item_features)
    
    mlflow.log_params(cb_model.get_params())
    
    cb_results = evaluator.evaluate_model(
        cb_model, test, n_items, model_name='Content-Based',
        use_idx=False
    )
    
    for metric_name, value in cb_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(metric_name.replace('@', '_at_'), value)
    
    with open(os.path.join(MODEL_DIR, 'content_based.pkl'), 'wb') as f:
        pickle.dump(cb_model, f)
    mlflow.log_artifact(os.path.join(MODEL_DIR, 'content_based.pkl'))
    
    print("\nContent-Based Results:")
    for k, v in sorted(cb_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")

## 5. Neural Collaborative Filtering (PyTorch)

In [ ]:
ncf_config = config['models']['ncf']

with mlflow.start_run(run_name='neural_collaborative_filtering'):
    ncf_trainer = NCFTrainer(n_users, n_items, ncf_config)
    ncf_trainer.fit(train, val, epochs=ncf_config['epochs'])
    
    mlflow.log_params(ncf_trainer.get_params())
    
    # Log training curves
    for epoch, loss in enumerate(ncf_trainer.train_losses):
        mlflow.log_metric('train_loss', loss, step=epoch)
    for epoch, loss in enumerate(ncf_trainer.val_losses):
        mlflow.log_metric('val_loss', loss, step=epoch)
    
    ncf_results = evaluator.evaluate_model(
        ncf_trainer, test, n_items, model_name='NCF',
        user_idx_col='user_idx', use_idx=True
    )
    
    for metric_name, value in ncf_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(metric_name.replace('@', '_at_'), value)
    
    ncf_trainer.save_model(os.path.join(MODEL_DIR, 'ncf_model.pt'))
    mlflow.log_artifact(os.path.join(MODEL_DIR, 'ncf_model.pt'))
    
    # Plot training curves
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ncf_trainer.train_losses, label='Train Loss', linewidth=2)
    if ncf_trainer.val_losses:
        ax.plot(ncf_trainer.val_losses, label='Val Loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.set_title('NCF Training Curves')
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(MODEL_DIR, 'ncf_training_curves.png'), dpi=150)
    mlflow.log_artifact(os.path.join(MODEL_DIR, 'ncf_training_curves.png'))
    plt.show()
    
    print("\nNCF Results:")
    for k, v in sorted(ncf_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")

## 6. Hybrid Ensemble

In [ ]:
with mlflow.start_run(run_name='hybrid_ensemble'):
    hybrid = HybridRecommender(
        weights=config['models']['hybrid']['weights']
    )
    hybrid.set_mappings(loader.idx_to_item, loader.item_to_idx)
    hybrid.register_model('collaborative', cf_model)
    hybrid.register_model('matrix_factorization', mf_model)
    hybrid.register_model('content_based', cb_model)
    hybrid.register_model('ncf', ncf_trainer)
    
    mlflow.log_params(hybrid.get_params())
    
    # Evaluate hybrid on test users
    from collections import defaultdict
    from src.evaluation import precision_at_k, recall_at_k, ndcg_at_k, hit_rate_at_k
    
    test_gt = defaultdict(set)
    for _, row in test.iterrows():
        test_gt[row['visitorid']].add(row['itemid'])
    
    hybrid_metrics = {k: {'precision': [], 'recall': [], 'ndcg': [], 'hit_rate': []} 
                      for k in config['evaluation']['top_k']}
    
    for user_id in list(test_gt.keys())[:500]:
        user_idx = loader.user_to_idx.get(user_id)
        recs = hybrid.predict_for_user(user_id, user_idx, top_n=max(config['evaluation']['top_k']))
        rec_items = [item_id for item_id, _ in recs]
        relevant = test_gt[user_id]
        
        for k in config['evaluation']['top_k']:
            hybrid_metrics[k]['precision'].append(precision_at_k(rec_items, relevant, k))
            hybrid_metrics[k]['recall'].append(recall_at_k(rec_items, relevant, k))
            hybrid_metrics[k]['ndcg'].append(ndcg_at_k(rec_items, relevant, k))
            hybrid_metrics[k]['hit_rate'].append(hit_rate_at_k(rec_items, relevant, k))
    
    hybrid_results = {}
    for k in config['evaluation']['top_k']:
        for metric_name in ['precision', 'recall', 'ndcg', 'hit_rate']:
            key = f'{metric_name}@{k}'
            value = np.mean(hybrid_metrics[k][metric_name])
            hybrid_results[key] = value
            mlflow.log_metric(key.replace('@', '_at_'), value)
    
    print("\nHybrid Ensemble Results:")
    for k, v in sorted(hybrid_results.items()):
        print(f"  {k}: {v:.4f}")

## 7. Model Comparison Summary

In [ ]:
all_results = {
    'CF-KNN': cf_results,
    'MF-SVD': mf_results,
    'Content-Based': cb_results,
    'NCF': ncf_results,
    'Hybrid': hybrid_results
}

comparison_df = evaluator.compare_models(all_results)
print("\n=== Model Comparison ===")
display(comparison_df)

# Visualization
metrics_to_plot = ['precision@10', 'recall@10', 'ndcg@10', 'hit_rate@10']
plot_data = {}
for metric in metrics_to_plot:
    plot_data[metric] = {name: res.get(metric, 0) for name, res in all_results.items()}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12']

for i, metric in enumerate(metrics_to_plot):
    models = list(plot_data[metric].keys())
    values = list(plot_data[metric].values())
    bars = axes[i].bar(models, values, color=colors)
    axes[i].set_title(metric.upper())
    axes[i].tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(MODEL_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()